In [1]:
%matplotlib inline
import json
import math
import statistics
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets

def _find_animacy_root(start: Path) -> Path:
    resolved = start.resolve()
    for base in (resolved, *resolved.parents):
        for candidate in (base, base / "animacy-circuit"):
            if (
                (candidate / "dataset").is_dir()
                and (candidate / "results").is_dir()
                and (candidate / "scripts").is_dir()
            ):
                return candidate
    raise FileNotFoundError("Could not locate the animacy-circuit project root.")

project_root = _find_animacy_root(Path.cwd())
working_dir = project_root

In [2]:
REQUIRED_KEYS = ("uid", "clean", "corrupt", "clean_p_yes", "corrupt_p_yes")

def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows: Iterable[dict]) -> int:
    rows = list(rows)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return len(rows)

def quantile(sorted_vals: list[float], q: float) -> float | None:
    if not sorted_vals:
        return None
    if len(sorted_vals) == 1:
        return float(sorted_vals[0])
    pos = (len(sorted_vals) - 1) * q
    low = int(math.floor(pos))
    high = int(math.ceil(pos))
    if low == high:
        return float(sorted_vals[low])
    frac = pos - low
    return float(sorted_vals[low] + (sorted_vals[high] - sorted_vals[low]) * frac)

def summarize(values: list[float]) -> dict[str, float | int | None]:
    if not values:
        return {"count": 0, "mean": None, "median": None, "std": None, "min": None, "max": None, "q05": None, "q25": None, "q75": None, "q95": None}
    sorted_vals = sorted(values)
    return {
        "count": len(values),
        "mean": float(statistics.mean(values)),
        "median": float(statistics.median(values)),
        "std": float(statistics.pstdev(values)) if len(values) > 1 else 0.0,
        "min": float(sorted_vals[0]),
        "max": float(sorted_vals[-1]),
        "q05": quantile(sorted_vals, 0.05),
        "q25": quantile(sorted_vals, 0.25),
        "q75": quantile(sorted_vals, 0.75),
        "q95": quantile(sorted_vals, 0.95),
    }

def _silverman_bandwidth(values: list[float]) -> float:
    if len(values) < 2: return 0.05
    std = statistics.pstdev(values)
    if std <= 0.0: return 0.05
    return max(1.06 * std * (len(values) ** (-0.2)), 0.01)

def _count_curve_bin_width(values: list[float], points: int) -> float:
    if len(values) < 2: return max(1.0 / max(points - 1, 1), 0.05)
    sorted_vals = sorted(values)
    q25 = quantile(sorted_vals, 0.25)
    q75 = quantile(sorted_vals, 0.75)
    if q25 is not None and q75 is not None:
        iqr = q75 - q25
        if iqr > 0.0:
            return max(2.0 * iqr * (len(values) ** (-1.0 / 3.0)), 1.0 / max(points - 1, 1))
    std = statistics.pstdev(values)
    if std > 0.0:
        return max(3.49 * std * (len(values) ** (-1.0 / 3.0)), 1.0 / max(points - 1, 1))
    return max(1.0 / max(points - 1, 1), 0.05)

def _kde_curve(values: list[float], points: int, bandwidth: float) -> tuple[list[float], list[float]]:
    x_values = [i / (points - 1) for i in range(points)]
    normalizer = 1.0 / (math.sqrt(2.0 * math.pi) * bandwidth * len(values))
    y_values: list[float] = []
    for x in x_values:
        kernel_sum = 0.0
        for v in values:
            z = (x - v) / bandwidth
            kernel_sum += math.exp(-0.5 * z * z)
        y_values.append(normalizer * kernel_sum)
    return x_values, y_values

def _smoothed_count_curve(values: list[float], points: int, bandwidth: float) -> tuple[list[float], list[float]]:
    x_values, density_values = _kde_curve(values, points=points, bandwidth=bandwidth)
    count_bin_width = _count_curve_bin_width(values, points=points)
    y_values = [density * len(values) * count_bin_width for density in density_values]
    return x_values, y_values

def normalize_scored_row(row: dict) -> dict:
    missing = [key for key in REQUIRED_KEYS if key not in row]
    if missing:
        raise ValueError(f"Missing required keys {missing}")
    normalized = dict(row)
    normalized["uid"] = str(row["uid"])
    normalized["clean_p_yes"] = float(row["clean_p_yes"])
    normalized["corrupt_p_yes"] = float(row["corrupt_p_yes"])
    return normalized

In [3]:
base_dir = Path(working_dir) / "dataset" / "semantic_meaningful"
plot_dir = base_dir / "prefix_score_plots"

CONFIG = {
    "merged_path": base_dir / "scored_semantic_pairs.jsonl",
    "filtered_out": base_dir / "accepted_filtered_pairs.jsonl",
    "rejected_out": base_dir / "rejected_filtered_pairs.jsonl",
    "analysis_json": base_dir / "prefix_score_analysis.json",
    
    # Static plots will be saved here upon export
    "plot_clean": plot_dir / "clean_p_yes_distribution.png",
    "plot_corrupt": plot_dir / "corrupt_p_yes_distribution.png",
    
    "curve_points": 400,
    "bandwidth": 0.0,
    "strict_mode": True
}

In [4]:
rows = load_jsonl(CONFIG["merged_path"])
parsed_rows = []
malformed_rows = 0
malformed_messages = []

for idx, row in enumerate(rows, start=1):
    try:
        parsed_rows.append(normalize_scored_row(row))
    except Exception as exc:
        malformed_rows += 1
        if len(malformed_messages) < 10:
            malformed_messages.append(f"row {idx}: {exc}")
        if CONFIG["strict_mode"]:
            raise ValueError(f"Strict mode validation failed: malformed_rows={malformed_rows} | msg: {exc}")

print(f"Loaded {len(parsed_rows):,} valid rows.")

clean_values = [r["clean_p_yes"] for r in parsed_rows]
corrupt_values = [r["corrupt_p_yes"] for r in parsed_rows]

Loaded 20,000 valid rows.


In [5]:
# Pre-calculate smoothed count curves so the slider is perfectly smooth and doesn't lag
bw_clean = CONFIG["bandwidth"] if CONFIG["bandwidth"] > 0.0 else _silverman_bandwidth(clean_values)
x_clean, y_clean = _smoothed_count_curve(clean_values, points=CONFIG["curve_points"], bandwidth=bw_clean)

bw_corrupt = CONFIG["bandwidth"] if CONFIG["bandwidth"] > 0.0 else _silverman_bandwidth(corrupt_values)
x_corrupt, y_corrupt = _smoothed_count_curve(corrupt_values, points=CONFIG["curve_points"], bandwidth=bw_corrupt)

def interactive_thresholds(clean_min, corrupt_min):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Clean Plot ---
    ax1.plot(x_clean, y_clean, color="#1f77b4", linewidth=2.2)
    ax1.fill_between(x_clean, y_clean, color="#1f77b4", alpha=0.15)
    
    # Shade accepted area
    x_c_shade = [x for x in x_clean if x >= clean_min]
    y_c_shade = [y for x, y in zip(x_clean, y_clean) if x >= clean_min]
    if x_c_shade:
        ax1.fill_between(x_c_shade, y_c_shade, color="#2ca02c", alpha=0.4)
        
    ax1.axvline(x=clean_min, color='red', linestyle='--', linewidth=2)
    
    # Calculate exact samples
    samples_clean = sum(1 for v in clean_values if v >= clean_min)
    pct_clean = (samples_clean / len(clean_values)) * 100
    
    ax1.set_title("Clean Prefix Scores")
    ax1.set_xlabel("clean_p_yes")
    ax1.set_ylabel("Smoothed count")
    ax1.set_xlim(0, 1)
    ax1.text(0.05, 0.95, f"Threshold ≥ {clean_min:.2f}\nSamples: {samples_clean:,} ({pct_clean:.1f}%)", 
             transform=ax1.transAxes, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # --- Corrupt Plot ---
    ax2.plot(x_corrupt, y_corrupt, color="#ff7f0e", linewidth=2.2)
    ax2.fill_between(x_corrupt, y_corrupt, color="#ff7f0e", alpha=0.15)
    
    # Shade accepted area
    x_cor_shade = [x for x in x_corrupt if x >= corrupt_min]
    y_cor_shade = [y for x, y in zip(x_corrupt, y_corrupt) if x >= corrupt_min]
    if x_cor_shade:
        ax2.fill_between(x_cor_shade, y_cor_shade, color="#2ca02c", alpha=0.4)
        
    ax2.axvline(x=corrupt_min, color='red', linestyle='--', linewidth=2)
    
    # Calculate exact samples
    samples_corrupt = sum(1 for v in corrupt_values if v >= corrupt_min)
    pct_corrupt = (samples_corrupt / len(corrupt_values)) * 100
    
    ax2.set_title("Corrupt Prefix Scores")
    ax2.set_xlabel("corrupt_p_yes")
    ax2.set_xlim(0, 1)
    ax2.text(0.05, 0.95, f"Threshold ≥ {corrupt_min:.2f}\nSamples: {samples_corrupt:,} ({pct_corrupt:.1f}%)", 
             transform=ax2.transAxes, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()

# Link the sliders to the function
slider_clean = FloatSlider(min=0.0, max=1.0, step=0.01, value=0.70, description='Clean Min:')
slider_corrupt = FloatSlider(min=0.0, max=1.0, step=0.01, value=0.70, description='Corrupt Min:')

out = interact(interactive_thresholds, clean_min=slider_clean, corrupt_min=slider_corrupt)

interactive(children=(FloatSlider(value=0.7, description='Clean Min:', max=1.0, step=0.01), FloatSlider(value=…

In [6]:
# Grab the values directly from the interactive widgets
FINAL_CLEAN_MIN = slider_clean.value
FINAL_CORRUPT_MIN = slider_corrupt.value

print(f"Applying thresholds: Clean >= {FINAL_CLEAN_MIN}, Corrupt >= {FINAL_CORRUPT_MIN}")

filtered_rows = []
rejected_rows = []
rejection_counts = {"clean_low": 0, "corrupt_low": 0, "both_low": 0}

for row in parsed_rows:
    clean_ok = row["clean_p_yes"] >= FINAL_CLEAN_MIN
    corrupt_ok = row["corrupt_p_yes"] >= FINAL_CORRUPT_MIN
    
    out_row = dict(row)
    out_row.pop("rejection_reasons", None)

    if clean_ok and corrupt_ok:
        filtered_rows.append(out_row)
        continue

    reasons = []
    if not clean_ok: reasons.append("clean_low")
    if not corrupt_ok: reasons.append("corrupt_low")

    if len(reasons) == 2:
        rejection_counts["both_low"] += 1
    elif reasons:
        rejection_counts[reasons[0]] += 1

    out_row["rejection_reasons"] = reasons
    rejected_rows.append(out_row)

# Save the final static plot images
def save_static_plot(x, y, title, xlabel, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x, y, color="#1f77b4", linewidth=2.2)
    ax.fill_between(x, y, color="#1f77b4", alpha=0.20)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Smoothed count")
    ax.set_xlim(0.0, 1.0)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)

save_static_plot(x_clean, y_clean, "Distribution of clean_p_yes", "clean_p_yes", CONFIG["plot_clean"])
save_static_plot(x_corrupt, y_corrupt, "Distribution of corrupt_p_yes", "corrupt_p_yes", CONFIG["plot_corrupt"])

# Write Data
filtered_written = write_jsonl(CONFIG["filtered_out"], filtered_rows)
rejected_written = write_jsonl(CONFIG["rejected_out"], rejected_rows)

# Generate and save Analysis JSON
analysis = {
    "meta": {
        "script": "Jupyter Notebook Interactive Export",
        "clean_min": FINAL_CLEAN_MIN,
        "corrupt_min": FINAL_CORRUPT_MIN,
        "curve_points": CONFIG["curve_points"],
        "bandwidth": CONFIG["bandwidth"],
    },
    "counts": {
        "input_rows": len(rows),
        "valid_rows": len(parsed_rows),
        "filtered_rows": filtered_written,
        "rejected_rows": rejected_written,
        "acceptance_rate_over_valid": (filtered_written / len(parsed_rows)) if parsed_rows else None,
        "rejection_reasons": rejection_counts,
    },
    "stats": {
        "clean_p_yes": summarize(clean_values),
        "corrupt_p_yes": summarize(corrupt_values),
    },
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}

CONFIG["analysis_json"].parent.mkdir(parents=True, exist_ok=True)
with open(CONFIG["analysis_json"], "w", encoding="utf-8") as f:
    json.dump(analysis, f, indent=2)

print(f"✅ Success! Filtered rows written: {filtered_written:,}")

Applying thresholds: Clean >= 0.9, Corrupt >= 0.9
✅ Success! Filtered rows written: 16,305
